# Target Horizon Repair
Objective 1 — Trace where actual_return was introduced

In [1]:
import pandas as pd
from pathlib import Path

FINAL_DIR = Path("../Market_Data/final")

files = {
    "Step-33":   "step33_cost_aware_portfolio.parquet",
    "Step-34":   "step34_periodic_rebalancing.parquet",
    "Step-34.1": "step34_1_turnover_audit.parquet",
    "Step-34.2": "step34_2_persistence_engine.parquet",
    "Step-34.3": "step34_3_regime_persistence.parquet",
}

for name, fname in files.items():
    try:
        df = pd.read_parquet(FINAL_DIR / fname)
        print(f"\n{name}")
        print("Has actual_return:", "actual_return" in df.columns)
        print("Has target_future_return_t3:", "target_future_return_t3" in df.columns)

        if "actual_return" in df.columns:
            print(df["actual_return"].describe())

        if "target_future_return_t3" in df.columns:
            print(df["target_future_return_t3"].describe())

        if "actual_return" in df.columns and "target_future_return_t3" in df.columns:
            diff = (df["actual_return"] - df["target_future_return_t3"]).abs().mean()
            print(f"Mean diff: {diff:.6f}")
    except FileNotFoundError:
        print(f"{name}: file not found — skip")



Step-33
Has actual_return: True
Has target_future_return_t3: False
count    4900.000000
mean        0.002233
std         0.033101
min        -0.144255
25%        -0.015622
50%         0.001065
75%         0.018694
max         0.350119
Name: actual_return, dtype: float64
Step-34: file not found — skip
Step-34.1: file not found — skip

Step-34.2
Has actual_return: True
Has target_future_return_t3: False
count    4900.000000
mean        0.002233
std         0.033101
min        -0.144255
25%        -0.015622
50%         0.001065
75%         0.018694
max         0.350119
Name: actual_return, dtype: float64

Step-34.3
Has actual_return: True
Has target_future_return_t3: False
count    4900.000000
mean        0.000725
std         0.021259
min        -0.118721
25%        -0.009582
50%         0.000285
75%         0.010530
max         0.199958
Name: actual_return, dtype: float64


The exact notebook where `actual_return` first diverged from `target_future_return_t3` is Step 33. `target_future_return_t3` disappears from Step 33 onwards completely, and is replaced by `actual_return` which matches `target_future_return_t3` in Step 33, but diverges in Step 34.3.

Objective 2 — Direct IC proof before any remapping

In [2]:
df = pd.read_parquet(FINAL_DIR / "step34_3_regime_persistence.parquet")

ic_original = df.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
    include_groups=False
).mean()

try:
    ic_corrected = df.groupby("date", group_keys=False).apply(
        lambda x: x["pred_score"].corr(x["target_future_return_t3"], method="spearman"),
        include_groups=False
    ).mean()
except KeyError as e:
    print(f"KeyError: {e} - STOPPING ENTIRELY")
    ic_corrected = -999

print(f"Original IC  (pred_score vs actual_return):            {ic_original:.6f}")
print(f"Corrected IC (pred_score vs target_future_return_t3):  {ic_corrected:.6f}")
print(f"Expected:                                               0.026841")
print(f"Match: {'YES' if abs(ic_corrected - 0.026841) < 0.005 else 'NO'}")


c:\Users\Priyanshu\Desktop\Main\final-year\venv\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


KeyError: 'target_future_return_t3' - STOPPING ENTIRELY
Original IC  (pred_score vs actual_return):            -0.202964
Corrected IC (pred_score vs target_future_return_t3):  -999.000000
Expected:                                               0.026841
Match: NO


Objective 3 — Horizon identification via lag sweep

In [3]:
# Identify exactly what horizon actual_return represents
print("Lag sweep: actual_return vs target_future_return_t3")

try:
    df_sorted = df.sort_values(["ticker", "date"])

    for lag in range(-5, 6):
        shifted = df_sorted.groupby("ticker")["actual_return"].shift(lag)
        corr = shifted.corr(df_sorted["target_future_return_t3"])
        marker = " ← PEAK" if abs(corr) == max(
            abs(df_sorted.groupby("ticker")["actual_return"].shift(l)
                .corr(df_sorted["target_future_return_t3"])) for l in range(-5, 6)
        ) else ""
        print(f"Lag {lag:+d}: corr={corr:.4f}{marker}")
except KeyError as e:
    print(f"Cannot run lag sweep: {e}")


Lag sweep: actual_return vs target_future_return_t3
Cannot run lag sweep: 'target_future_return_t3'


Identified horizon: Cannot be identified because `target_future_return_t3` is completely missing.

Objective 4 — Rebuild with corrected target and save

In [4]:
# Assert target column is complete before remapping
if "target_future_return_t3" not in df.columns:
    print("HALT: target_future_return_t3 is completely missing from df.")
else:
    assert df["target_future_return_t3"].notna().all(), \
        "Missing values in target_future_return_t3 — cannot remap safely"

    # Remap in memory — do not modify original parquet
    df["actual_return"] = df["target_future_return_t3"]

    # Verify IC is restored
    ic_restored = df.groupby("date", group_keys=False).apply(
        lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
        include_groups=False
    ).mean()

    print(f"Restored IC: {ic_restored:.6f}")
    assert ic_restored > 0.020, f"IC not restored: {ic_restored:.4f}"
    assert abs(ic_restored - 0.026841) < 0.005, \
        f"IC does not match Step-33 ground truth: {ic_restored:.4f}"

    df.to_parquet(FINAL_DIR / "step34_3_corrected.parquet")
    print("Saved: step34_3_corrected.parquet")


HALT: target_future_return_t3 is completely missing from df.


In [5]:
ic_restored = -999.0

print(f"""
Target Horizon Repair — Status
================================
actual_return introduced at      : Step-33
actual_return represents         : [Unknown, missing reference column]
IC before fix                    : {ic_original:.4f}
IC after fix                     : {ic_restored:.4f}  (target: 0.026841)
IC matches Step-33 ground truth  : {'YES' if abs(ic_restored - 0.026841) < 0.005 else 'NO'}

Corrected file saved             : FAILED TO SAVE

Component Status:
  Step-28 Alpha     : ✅ Valid
  Step-33 Alpha     : ✅ Valid
  Signal lineage    : ✅ Preserved
  Step-34.3 target  : {'✅ Repaired' if abs(ic_restored - 0.026841) < 0.005 else '❌ Still broken'}
  Step-34.4         : {'✅ Re-enabled' if abs(ic_restored - 0.026841) < 0.005 else '⛔ Still paused'}

Safe to proceed to Step-34.4     : {'YES' if abs(ic_restored - 0.026841) < 0.005 else 'NO'}
""")



Target Horizon Repair — Status
actual_return introduced at      : Step-33
actual_return represents         : [Unknown, missing reference column]
IC before fix                    : -0.2030
IC after fix                     : -999.0000  (target: 0.026841)
IC matches Step-33 ground truth  : NO

Corrected file saved             : FAILED TO SAVE

Component Status:
  Step-28 Alpha     : ✅ Valid
  Step-33 Alpha     : ✅ Valid
  Signal lineage    : ✅ Preserved
  Step-34.3 target  : ❌ Still broken
  Step-34.4         : ⛔ Still paused

Safe to proceed to Step-34.4     : NO

